In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [4]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [5]:
spark.version

'3.5.5'

In [8]:
df = spark.read.json('/Users/dhruvik/Documents/spark_practice/data/diamonds.json')

In [10]:
df.show()

+-----+-------+-----+---------+-----+-----+
|carat|clarity|color|      cut|depth|price|
+-----+-------+-----+---------+-----+-----+
| 0.23|    SI2|    E|    Ideal| 61.5|  326|
| 0.21|    SI1|    E|  Premium| 59.8|  326|
| 0.23|    VS1|    E|     Good| 56.9|  327|
| 0.29|    VS2|    I|  Premium| 62.4|  334|
| 0.31|    SI2|    J|     Good| 63.3|  335|
| 0.24|   VVS2|    J|Very Good| 62.8|  336|
| 0.24|   VVS1|    I|Very Good| 62.3|  336|
| 0.26|    SI1|    H|Very Good| 61.9|  337|
| 0.22|    VS2|    E|     Fair| 65.1|  337|
| 0.23|    VS1|    H|Very Good| 59.4|  338|
|  0.3|    SI1|    J|     Good| 64.0|  339|
| 0.23|    VS1|    J|    Ideal| 62.8|  340|
| 0.22|    SI1|    F|  Premium| 60.4|  342|
| 0.31|    SI2|    J|    Ideal| 62.2|  344|
|  0.2|    SI2|    E|  Premium| 60.2|  345|
| 0.32|     I1|    E|  Premium| 60.9|  345|
|  0.3|    SI2|    I|    Ideal| 62.0|  348|
|  0.3|    SI1|    J|     Good| 63.4|  351|
|  0.3|    SI1|    J|     Good| 63.8|  351|
|  0.3|    SI1|    J|Very Good| 

In [11]:
df.count()

53940

In [13]:
raw_fire_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema","true")
    .load("/Users/dhruvik/Documents/spark_practice/data/sf-fire-calls.csv")
)

In [14]:
raw_fire_df.count()

175296

In [16]:
raw_fire_df.show()

+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------------+--------------------------+----------------------+------------------+--------------------+--------------------+-------------+---------+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|       AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|      UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|        RowID|    Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+

26/06/03 16:19:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [17]:
raw_fire_df.printSchema()

root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- WatchDate: string (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: string (nullable = true)
 |-- Box: string (nullable = true)
 |-- OriginalPriority: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- FinalPriority: integer (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true)
 |-- FirePreventionDistrict: string (nullable = true)
 

In [18]:
from pyspark.sql.functions import to_timestamp,expr

fire_df = raw_fire_df.withColumns(
    {
        "AvailableDtTm": to_timestamp("AvailableDtTm","dd/MM/yyyy hh:mm:ss a"),
        "Zipcode": expr("cast(Zipcode as string)"),
        "FinalPriority": expr("cast(FinalPriority as string)")
    }
)

In [19]:
fire_df.printSchema()

root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- WatchDate: string (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: timestamp (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: string (nullable = true)
 |-- Box: string (nullable = true)
 |-- OriginalPriority: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- FinalPriority: string (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true)
 |-- FirePreventionDistrict: string (nullable = true)


In [20]:
fire_df.write.mode("overwrite").option("path", "./elexi_spark_warehouse/sf_fire_calls").saveAsTable("sf_fire_calls")


26/06/03 16:19:16 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/03 16:19:16 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/06/03 16:19:16 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/03 16:19:16 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dhruvik@192.168.1.14
26/06/03 16:19:17 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/06/03 16:19:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/03 16:19:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/03 16:19:17 WARN MemoryManager: Total allocation exceeds 95.00% (

In [32]:
spark.sql("SELECT * FROM sf_fire_calls").show()

+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------------+--------------------------+----------------------+------------------+--------------------+--------------------+--------------+---------+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|      AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|      UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|         RowID|    Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-

In [35]:
fire_df = spark.read.table("sf_fire_calls")

In [47]:
df_1 = fire_df.select("CallType", "Zipcode")
df_2 = df_1.where("CallType is not null")
df_3 = df_2.groupBy("CallType", "Zipcode").count()
df_4 = df_3.orderBy("count", ascending=False)
df_5 = df_4.limit(3)

df_5.show()


+----------------+-------+-----+
|        CallType|Zipcode|count|
+----------------+-------+-----+
|Medical Incident|  94102|16130|
|Medical Incident|  94103|14775|
|Medical Incident|  94110| 9995|
+----------------+-------+-----+



In [49]:
df_5.explain(mode="extended")

== Parsed Logical Plan ==
GlobalLimit 3
+- LocalLimit 3
   +- Sort [count#973L DESC NULLS LAST], true
      +- Aggregate [CallType#464, Zipcode#471], [CallType#464, Zipcode#471, count(1) AS count#973L]
         +- Filter isnotnull(CallType#464)
            +- Project [CallType#464, Zipcode#471]
               +- SubqueryAlias spark_catalog.default.sf_fire_calls
                  +- Relation spark_catalog.default.sf_fire_calls[CallNumber#461,UnitID#462,IncidentNumber#463,CallType#464,CallDate#465,WatchDate#466,CallFinalDisposition#467,AvailableDtTm#468,Address#469,City#470,Zipcode#471,Battalion#472,StationArea#473,Box#474,OriginalPriority#475,Priority#476,FinalPriority#477,ALSUnit#478,CallTypeGroup#479,NumAlarms#480,UnitType#481,UnitSequenceInCallDispatch#482,FirePreventionDistrict#483,SupervisorDistrict#484,... 4 more fields] parquet

== Analyzed Logical Plan ==
CallType: string, Zipcode: string, count: bigint
GlobalLimit 3
+- LocalLimit 3
   +- Sort [count#973L DESC NULLS LAST], true


In [89]:
result_df = fire_df.select("CallType", "Zipcode").where("CallType is not null").groupBy("CallType", "ZipCode").count().orderBy("count", descending=True).limit(3)

In [91]:
result_df.show()

+---------+-------+-----+
| CallType|ZipCode|count|
+---------+-------+-----+
|Oil Spill|  94118|    1|
|   HazMat|  94134|    1|
|Oil Spill|  94117|    1|
+---------+-------+-----+



In [59]:
q1_df = (fire_df.selectExpr("count(distinct CallType) as distinct_call_type_count"))

In [61]:
q1_df.show()

+------------------------+
|distinct_call_type_count|
+------------------------+
|                      30|
+------------------------+



In [63]:
q2_df = (fire_df.where("CallType is not null").selectExpr("CallType as distinct_call_type").distinct())

In [65]:
q2_df.show()

+--------------------+
|  distinct_call_type|
+--------------------+
|Elevator / Escala...|
|  Aircraft Emergency|
|      Administrative|
|              Alarms|
|Odor (Strange / U...|
|Citizen Assist / ...|
|              HazMat|
|Watercraft in Dis...|
|           Explosion|
|           Oil Spill|
|        Vehicle Fire|
|  Suspicious Package|
|               Other|
|        Outside Fire|
|   Traffic Collision|
|       Assist Police|
|Gas Leak (Natural...|
|        Water Rescue|
|   Electrical Hazard|
|      Structure Fire|
+--------------------+
only showing top 20 rows



In [71]:
q3_df = (fire_df.where("Delay>10").select("CallNumber","Delay"))
q3_df.count()

5393

In [75]:
q3 = fire_df.where("Delay>5").select("CallNumber","Delay")
q3.show()

+----------+---------+
|CallNumber|    Delay|
+----------+---------+
|  10680052|13.483334|
|  10680059|      5.6|
|  10680355|    12.25|
|  10680401|      5.2|
|  10690009| 7.733333|
|  10690039| 5.516667|
|  10700142| 6.016667|
|  10710341|6.8333335|
|  10720011| 7.633333|
|  10720282| 5.016667|
|  10720334| 5.516667|
|  10720448| 7.883333|
|  10730135| 9.916667|
|  10730135| 9.283334|
|  10730243|11.316667|
|  10740292|6.2166667|
|  10740335|      9.4|
|  10750017| 7.366667|
|  10750033|     21.7|
|  10750033|     7.85|
+----------+---------+
only showing top 20 rows



In [77]:
q4_df = (
    fire_df.select("CallType")
    .where("CallType is not null")
    .groupBy("CallType")
    .count()
    .orderBy("count", ascending=False)
)
q4_df.show()

+--------------------+------+
|            CallType| count|
+--------------------+------+
|    Medical Incident|113794|
|      Structure Fire| 23319|
|              Alarms| 19406|
|   Traffic Collision|  7013|
|Citizen Assist / ...|  2524|
|               Other|  2166|
|        Outside Fire|  2094|
|        Vehicle Fire|   854|
|Gas Leak (Natural...|   764|
|        Water Rescue|   755|
|Odor (Strange / U...|   490|
|   Electrical Hazard|   482|
|Elevator / Escala...|   453|
|Smoke Investigati...|   391|
|          Fuel Spill|   193|
|              HazMat|   124|
|Industrial Accidents|    94|
|           Explosion|    89|
|Train / Rail Inci...|    57|
|  Aircraft Emergency|    36|
+--------------------+------+
only showing top 20 rows



In [2]:
flight_time_raw_df = (
    spark.read
    .format("json")
    .option("mode", "FAILFAST")
    .option("dateFormat", "M/d/yyyy")
    .load("/Users/dhruvik/Documents/spark_practice/data/flight-time.json")
)

In [4]:
flight_time_raw_df.limit(3).show()

+--------+---------+------------+------------+--------+----+--------------+--------+--------+----------+-----------------+------+----------------+-------+---------+
|ARR_TIME|CANCELLED|CRS_ARR_TIME|CRS_DEP_TIME|DEP_TIME|DEST|DEST_CITY_NAME|DISTANCE| FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|TAXI_IN|WHEELS_ON|
+--------+---------+------------+------------+--------+----+--------------+--------+--------+----------+-----------------+------+----------------+-------+---------+
|    1348|        0|        1400|        1115|    1113| ATL|   Atlanta, GA|     946|1/1/2000|        DL|             1451|   BOS|      Boston, MA|      5|     1343|
|    1543|        0|        1559|        1315|    1311| ATL|   Atlanta, GA|     946|1/1/2000|        DL|             1479|   BOS|      Boston, MA|      7|     1536|
|    1651|        0|        1721|        1415|    1414| ATL|   Atlanta, GA|     946|1/1/2000|        DL|             1857|   BOS|      Boston, MA|      9|     1642|
+--------+

In [6]:
flight_time_raw_df.printSchema()

root
 |-- ARR_TIME: long (nullable = true)
 |-- CANCELLED: long (nullable = true)
 |-- CRS_ARR_TIME: long (nullable = true)
 |-- CRS_DEP_TIME: long (nullable = true)
 |-- DEP_TIME: long (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DISTANCE: long (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: long (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- TAXI_IN: long (nullable = true)
 |-- WHEELS_ON: long (nullable = true)



In [8]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField

flight_schema = StructType([
    StructField("FL_DATE", DateType()),
    StructField("OP_CARRIER", StringType()),
    StructField("OP_CARRIER_FL_NUM", StringType()),
    StructField("ORIGIN", StringType()),
    StructField("ORIGIN_CITY_NAME", StringType()),
    StructField("DEST", StringType()),
    StructField("DEST_CITY_NAME", StringType()),
    StructField("CRS_DEP_TIME", LongType()),
    StructField("DEP_TIME", LongType()),
    StructField("WHEELS_ON", IntegerType()),
    StructField("TAXI_IN", IntegerType()),
    StructField("CRS_ARR_TIME", LongType()),
    StructField("ARR_TIME", LongType()),
    StructField("CANCELLED", IntegerType()),
    StructField("DISTANCE", IntegerType())
])

In [10]:
flight_time_raw_with_schema_df = (
    spark.read
    .format("json")
    .option("mode", "FAILFAST")
    .option("dateFormat", "M/d/yyyy")
    .schema(flight_schema)
    .load("/Users/dhruvik/Documents/spark_practice/data/flight-time.json")
)

In [12]:
flight_time_raw_with_schema_df.limit(3).show()

+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|
+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|
|2000-01-01|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|        1559|    1543|        0|     946|
|2000-01-01|        DL|             1857|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1415|    1414|     1642|      9|        1721|    1651|        0|     946

In [14]:
flight_time_raw_with_schema_df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- CRS_DEP_TIME: long (nullable = true)
 |-- DEP_TIME: long (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- CRS_ARR_TIME: long (nullable = true)
 |-- ARR_TIME: long (nullable = true)
 |-- CANCELLED: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)



In [16]:
flight_time_raw_with_schema_df.write.mode("overwrite").saveAsTable("flight_time_raw")

26/06/04 16:26:33 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/04 16:26:33 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/06/04 16:26:34 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/04 16:26:34 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dhruvik@192.168.1.14
26/06/04 16:26:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/04 16:26:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/04 16:26:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/04 16:26:36 W

In [18]:
flight_time_raw_df = spark.read.table("flight_time_raw")

In [20]:
flight_time_raw_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|
+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|
|2000-01-01|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|        1559|    1543|        0|     946|
|2000-01-01|        DL|             1857|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1415|    1414|     1642|      9|        1721|    1651|        0|     946

In [36]:
from pyspark.sql.functions import expr

step_1_df = flight_time_raw_df.withColumns({
  "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'),2)"),
  "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
})

step_2_df = (
    step_1_df.withColumns({
      "CRS_DEP_TIME_NEW":expr("cast(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)")  
    })
)

In [38]:
step_2_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+--------------------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|CRS_DEP_TIME_HH|CRS_DEP_TIME_MM|    CRS_DEP_TIME_NEW|
+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+--------------------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|             11|             15|INTERVAL '11:15' ...|
|2000-01-01|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|   

In [40]:
def get_interval(hhmm_value):
    from pyspark.sql.functions import expr

    return expr(f"""
        cast(concat(left(lpad({hhmm_value},4,'0'),2), ':',
        right(lpad({hhmm_value},4,'0'),2))
        AS INTERVAL HOUR TO MINUTE)
    """)

In [42]:
result_df = (
    flight_time_raw_df.withColumns({
      "CRS_DEP_TIME":get_interval("CRS_DEP_TIME"),
      "DEP_TIME": get_interval("DEP_TIME"),
        "WHEELS_ON": get_interval("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
        "ARR_TIME": get_interval("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

In [46]:
result_df.write.mode("overwrite").saveAsTable("flight_time")

26/06/04 16:55:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/04 16:55:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/04 16:55:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/04 16:55:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/04 16:55:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/04 16:55:38 WARN HiveExternalCatalog: Hive incompatible types found: interval hour to minute, interval hour to minute, interval hour to minute, interval minute, interval hour to minute, interval hour to minute. Persist

In [48]:
flight_time_df = spark.read.table("flight_time")


In [50]:
flight_time_1_df = (
    flight_time_df.selectExpr(
        "fl_date as dep_date",
        "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
        "dep_date + crs_dep_time as crs_dep_time",
        "dep_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time"
    )
)

In [52]:
flight_time_1_df.show()

+----------+----------+-------------------+-------------------+-------------------+-------------------+
|  dep_date|  arr_date|       crs_dep_time|           dep_time|       crs_arr_time|           arr_time|
+----------+----------+-------------------+-------------------+-------------------+-------------------+
|2000-01-01|2000-01-02|2000-01-01 11:15:00|2000-01-01 11:13:00|2000-01-02 14:00:00|2000-01-02 13:48:00|
|2000-01-01|2000-01-02|2000-01-01 13:15:00|2000-01-01 13:11:00|2000-01-02 15:59:00|2000-01-02 15:43:00|
|2000-01-01|2000-01-02|2000-01-01 14:15:00|2000-01-01 14:14:00|2000-01-02 17:21:00|2000-01-02 16:51:00|
|2000-01-01|2000-01-02|2000-01-01 17:15:00|2000-01-01 17:20:00|2000-01-02 20:13:00|2000-01-02 20:05:00|
|2000-01-01|2000-01-02|2000-01-01 20:15:00|2000-01-01 20:10:00|2000-01-02 23:00:00|2000-01-02 22:40:00|
|2000-01-01|2000-01-01|2000-01-01 06:50:00|2000-01-01 06:49:00|2000-01-01 09:55:00|2000-01-01 10:03:00|
|2000-01-01|2000-01-02|2000-01-01 14:40:00|2000-01-01 14:46:00|2

In [62]:
from pyspark.sql.functions import to_date, col

flight_time_2_df = (
    flight_time_df.withColumnRenamed("fl_date", "dep_date")
    .withColumn("arr_date", expr("to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date"))
    .withColumns({
        "crs_dep_time": col("dep_date") + col("crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": expr("arr_date + arr_time"),
    })
)

In [64]:
flight_time_2_df.where("dep_date= '2000-01-01'").show()

+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+---------+--------+----------+
|  dep_date|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|       crs_dep_time|           dep_time|           WHEELS_ON|             TAXI_IN|       crs_arr_time|           arr_time|CANCELLED|DISTANCE|  arr_date|
+----------+----------+-----------------+------+----------------+----+--------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+---------+--------+----------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|2000-01-01 11:15:00|2000-01-01 11:13:00|INTERVAL '13:43' ...|INTERVAL '05' MINUTE|2000-01-02 14:00:00|2000-01-02 13:48:00|        0|     946|2000-01-02|
|2000-01-01|        DL|             1479|   BOS|

In [66]:
data_schema = "id int, source string, destination string, distance int"

In [68]:
data_list = [
    (101, "Mumbai", "Goa", 587),
    (102, "Mumbai", "Bangalore", 985),
    (102, "Mumbai", "Bangalore", 985),
    (103, "Delhi", "Chennai", 2208),
    (104, "Delhi", "Chennai", 2208),
    (105, "Bangalore", "Kolkata", 1868),
    (105, "Bangalore", "Kolkata", 1868)   
]

In [70]:
df = spark.createDataFrame(data=data_list, schema=data_schema)

In [74]:
display(df)

DataFrame[id: int, source: string, destination: string, distance: int]

In [76]:
df.show()

+---+---------+-----------+--------+
| id|   source|destination|distance|
+---+---------+-----------+--------+
|101|   Mumbai|        Goa|     587|
|102|   Mumbai|  Bangalore|     985|
|102|   Mumbai|  Bangalore|     985|
|103|    Delhi|    Chennai|    2208|
|104|    Delhi|    Chennai|    2208|
|105|Bangalore|    Kolkata|    1868|
|105|Bangalore|    Kolkata|    1868|
+---+---------+-----------+--------+



In [80]:
df.filter("source = 'Mumbai' and destination = 'Bangalore'").show()

+---+------+-----------+--------+
| id|source|destination|distance|
+---+------+-----------+--------+
|102|Mumbai|  Bangalore|     985|
|102|Mumbai|  Bangalore|     985|
+---+------+-----------+--------+



In [82]:
df.filter("source='Mumbai'").filter("destination = 'Bangalore'").show()

+---+------+-----------+--------+
| id|source|destination|distance|
+---+------+-----------+--------+
|102|Mumbai|  Bangalore|     985|
|102|Mumbai|  Bangalore|     985|
+---+------+-----------+--------+



In [84]:
df.filter((col("source")=='Mumbai')).show()

+---+------+-----------+--------+
| id|source|destination|distance|
+---+------+-----------+--------+
|101|Mumbai|        Goa|     587|
|102|Mumbai|  Bangalore|     985|
|102|Mumbai|  Bangalore|     985|
+---+------+-----------+--------+



In [86]:
df.filter((df.source=='Mumbai')).show()

+---+------+-----------+--------+
| id|source|destination|distance|
+---+------+-----------+--------+
|101|Mumbai|        Goa|     587|
|102|Mumbai|  Bangalore|     985|
|102|Mumbai|  Bangalore|     985|
+---+------+-----------+--------+



In [88]:
df.distinct().show()

+---+---------+-----------+--------+
| id|   source|destination|distance|
+---+---------+-----------+--------+
|101|   Mumbai|        Goa|     587|
|102|   Mumbai|  Bangalore|     985|
|103|    Delhi|    Chennai|    2208|
|104|    Delhi|    Chennai|    2208|
|105|Bangalore|    Kolkata|    1868|
+---+---------+-----------+--------+



In [90]:
df.dropDuplicates(["id", "source", "destination"]).show()

+---+---------+-----------+--------+
| id|   source|destination|distance|
+---+---------+-----------+--------+
|101|   Mumbai|        Goa|     587|
|102|   Mumbai|  Bangalore|     985|
|103|    Delhi|    Chennai|    2208|
|104|    Delhi|    Chennai|    2208|
|105|Bangalore|    Kolkata|    1868|
+---+---------+-----------+--------+



In [92]:
schema = "id int, name string, age string, salary double"

In [98]:
data_list = [
    (100, "Dhruvik", 45, 45000.00),
    (100, "Tarun", 36, 33000.00),
    (102, "David", 48, 28000.00)
]

In [100]:
#spark.createDataFrame(data=data_list).toDF("id", "name", "age", "salary")
sample_df = spark.createDataFrame(data=data_list, schema=schema)

In [102]:
sample_df.show()

+---+-------+---+-------+
| id|   name|age| salary|
+---+-------+---+-------+
|100|Dhruvik| 45|45000.0|
|100|  Tarun| 36|33000.0|
|102|  David| 48|28000.0|
+---+-------+---+-------+



In [104]:
from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumn("increment", expr("case when salary > 30000 then 3000 else salary * 10/100 end"))
    .withColumn("salary", expr("salary + increment"))
)

In [106]:
salary_df.show()

+---+-------+---+-------+---------+
| id|   name|age| salary|increment|
+---+-------+---+-------+---------+
|100|Dhruvik| 45|48000.0|   3000.0|
|100|  Tarun| 36|36000.0|   3000.0|
|102|  David| 48|30800.0|   2800.0|
+---+-------+---+-------+---------+



In [108]:
salary_df = salary_df.drop("age")

In [110]:
salary_df.show()

+---+-------+-------+---------+
| id|   name| salary|increment|
+---+-------+-------+---------+
|100|Dhruvik|48000.0|   3000.0|
|100|  Tarun|36000.0|   3000.0|
|102|  David|30800.0|   2800.0|
+---+-------+-------+---------+



In [19]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"),(2, "2022-05-19T11:30:10.0000")]

In [21]:
df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))

In [23]:
df_1.show(truncate=False)

+---+------------------------+
|id |string_time             |
+---+------------------------+
|1  |2022-05-18T10:30:30.0000|
|2  |2022-05-19T11:30:10.0000|
+---+------------------------+



In [25]:
from pyspark.sql.functions import to_timestamp

df_1.select("string_time", to_timestamp("string_time", "yyyy-MM-dd'T'HH:mm:ss.SSSS").alias("valid_time")).show()

+--------------------+-------------------+
|         string_time|         valid_time|
+--------------------+-------------------+
|2022-05-18T10:30:...|2022-05-18 10:30:30|
|2022-05-19T11:30:...|2022-05-19 11:30:10|
+--------------------+-------------------+



In [27]:
data_list_2 = [(1, "2022-05-18 10:30:30.0000"),(2, "19-05-2022 10:30:10.0000")]
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_2.show(truncate=False)

+---+------------------------+
|id |string_time             |
+---+------------------------+
|1  |2022-05-18 10:30:30.0000|
|2  |19-05-2022 10:30:10.0000|
+---+------------------------+



In [29]:
df_2.select("string_time", to_timestamp("string_time", "yyyy-MM-dd HH:mm:ss.SSSS").alias("valid_time")).show(truncate=False)

+------------------------+-------------------+
|string_time             |valid_time         |
+------------------------+-------------------+
|2022-05-18 10:30:30.0000|2022-05-18 10:30:30|
|19-05-2022 10:30:10.0000|NULL               |
+------------------------+-------------------+



In [31]:
spark.conf.get("spark.sql.session.timeZone")

'Asia/Kolkata'

In [33]:
spark.conf.set("spark.sql.session.timeZone", 'Etc/UTC')

In [35]:
spark.conf.get("spark.sql.session.timeZone")

'Etc/UTC'

In [37]:
event_ntz_schema = "component string, event_time string, reading string"

event_ntz_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(event_ntz_schema)
        .load("/Users/dhruvik/Documents/spark_practice/data/machine-events-no-tz.csv")
)

In [39]:
event_ntz_df.show(truncate=False)

+---------+-----------------------+-------+
|component|event_time             |reading|
+---------+-----------------------+-------+
|AXT594   |17-05-2022 06:14:10.359|23     |
|AXT594   |17-05-2022 06:14:25.380|25     |
|AXT594   |17-05-2022 06:14:35.346|21     |
|AXT594   |17-05-2022 06:14:45.381|22     |
|AXT594   |17-05-2022 06:14:55.356|25     |
|AXT594   |17-05-2022 06:15:05.372|23     |
|AXT594   |17-05-2022 06:15:15.355|24     |
|AXT594   |17-05-2022 06:16:25.326|24     |
|AXT594   |17-05-2022 06:17:35.345|21     |
|AXT594   |17-05-2022 06:18:45.365|22     |
+---------+-----------------------+-------+



In [41]:
from pyspark.sql.functions import to_timestamp_ntz, lit

event_valid_ntz_df = (
    event_ntz_df.withColumn("event_time_valid_ntz", to_timestamp_ntz("event_time", lit("dd-MM-yyyy HH:mm:ss.SSS")))
    )

In [43]:
stz = spark.conf.get("spark.sql.session.timeZone")

In [45]:
stz

'Etc/UTC'

In [47]:
from pyspark.sql.functions import convert_timezone, lit

events_df = (
    event_valid_ntz_df.withColumn("event_time_tz", to_timestamp(convert_timezone(lit("IST"), lit(stz), "event_time_valid_ntz")))
)

In [49]:
events_df.show(truncate=False)

+---------+-----------------------+-------+-----------------------+-----------------------+
|component|event_time             |reading|event_time_valid_ntz   |event_time_tz          |
+---------+-----------------------+-------+-----------------------+-----------------------+
|AXT594   |17-05-2022 06:14:10.359|23     |2022-05-17 06:14:10.359|2022-05-17 00:44:10.359|
|AXT594   |17-05-2022 06:14:25.380|25     |2022-05-17 06:14:25.38 |2022-05-17 00:44:25.38 |
|AXT594   |17-05-2022 06:14:35.346|21     |2022-05-17 06:14:35.346|2022-05-17 00:44:35.346|
|AXT594   |17-05-2022 06:14:45.381|22     |2022-05-17 06:14:45.381|2022-05-17 00:44:45.381|
|AXT594   |17-05-2022 06:14:55.356|25     |2022-05-17 06:14:55.356|2022-05-17 00:44:55.356|
|AXT594   |17-05-2022 06:15:05.372|23     |2022-05-17 06:15:05.372|2022-05-17 00:45:05.372|
|AXT594   |17-05-2022 06:15:15.355|24     |2022-05-17 06:15:15.355|2022-05-17 00:45:15.355|
|AXT594   |17-05-2022 06:16:25.326|24     |2022-05-17 06:16:25.326|2022-05-17 00

In [51]:
facilities_schema = """facility_id INT, facility_name STRING, member_cost DOUBLE, guest_cost DOUBLE, 
    initial_outlay DOUBLE, monthly_maintainance DOUBLE"""

members_schema = """member_id INT, last_name STRING, first_name STRING, address STRING, zip_code STRING, 
    telephone STRING, recommended_by STRING, joining_date DATE"""

bookings_schema = "booking_id INT, facility_id INT, member_id INT, start_time TIMESTAMP, slots INT"


facilities_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(facilities_schema)
        .load("/Users/dhruvik/Documents/spark_practice/data/facilities.csv")
)

facilities_df.write.mode("overwrite").saveAsTable("facilities")

26/06/05 16:54:58 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/05 16:54:58 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/06/05 16:54:59 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/05 16:54:59 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dhruvik@127.0.0.1
26/06/05 16:55:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: facid, fac_name, membercost, guestcost, initialoutlay, monthlymaintenance
 Schema: facility_id, facility_name, member_cost, guest_cost, initial_outlay, monthly_maintainance
Expected: facility_id but found: facid
CSV file: file:///Users/dhruvik/Documents/spark_practice/data/facilities.csv
26/06/05 16:55:00 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorizat

In [53]:
members_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(members_schema)
        .load("/Users/dhruvik/Documents/spark_practice/data/members.csv")
)

members_df.write.mode("overwrite").saveAsTable("members")

26/06/05 16:55:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: memid, surname, firstname, address, zipcode, telephone, recommendedby, joindate
 Schema: member_id, last_name, first_name, address, zip_code, telephone, recommended_by, joining_date
Expected: member_id but found: memid
CSV file: file:///Users/dhruvik/Documents/spark_practice/data/members.csv


In [55]:
bookings_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(bookings_schema)
        .load("/Users/dhruvik/Documents/spark_practice/data/bookings.csv")
)

bookings_df.write.mode("overwrite").saveAsTable("bookings")

26/06/05 16:55:24 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: bookid, facid, memid, starttime, slots
 Schema: booking_id, facility_id, member_id, start_time, slots
Expected: booking_id but found: bookid
CSV file: file:///Users/dhruvik/Documents/spark_practice/data/bookings.csv


In [62]:
bookings_df = spark.table("bookings").alias("b")
members_df = spark.table("members").alias("m")
facilities_df = spark.table("facilities").alias("f")

In [66]:
from pyspark.sql.functions import expr, col

join_expr = col("m.member_id") == col("b.member_id")

reports_df= (
    members_df.join(bookings_df, join_expr, "inner")
    .filter("m.last_name = 'Smith' and b.slots>5 ")
    .select("m.member_id", "m.first_name", "m.last_name", "b.facility_id","b.slots","b.start_time")
    .orderBy("m.first_name", col("b.slots").desc())
)

In [68]:
reports_df.show(truncate=False)

+---------+----------+---------+-----------+-----+-------------------+
|member_id|first_name|last_name|facility_id|slots|start_time         |
+---------+----------+---------+-----------+-----+-------------------+
|1        |Darren    |Smith    |2          |9    |2022-08-28 13:30:00|
|1        |Darren    |Smith    |2          |6    |2022-07-09 09:00:00|
|1        |Darren    |Smith    |2          |6    |2022-07-27 12:00:00|
|1        |Darren    |Smith    |2          |6    |2022-07-29 12:00:00|
|1        |Darren    |Smith    |2          |6    |2022-08-01 09:30:00|
|1        |Darren    |Smith    |2          |6    |2022-08-07 09:00:00|
|1        |Darren    |Smith    |2          |6    |2022-08-20 15:00:00|
|1        |Darren    |Smith    |2          |6    |2022-09-07 14:00:00|
|1        |Darren    |Smith    |2          |6    |2022-09-09 13:00:00|
|1        |Darren    |Smith    |2          |6    |2022-09-10 09:00:00|
|1        |Darren    |Smith    |2          |6    |2022-09-30 14:00:00|
|14   

In [70]:
report_df = (
    bookings_df
    .join(members_df, expr("b.member_id == m.member_id"), "inner")
    .join(facilities_df, col("b.facility_id") == col("f.facility_id"), "inner")
    .filter("m.last_name == 'Smith' and b.slots > 5")
    .selectExpr("m.member_id", "m.first_name", "m.last_name", "f.facility_name", "b.slots", "b.slots * f.member_cost as booking_amount", "b.start_time")
    .orderBy(col("m.first_name").asc(),  col("booking_amount").desc())
)

In [72]:
report_df.show(truncate=False)

+---------+----------+---------+---------------+-----+--------------+-------------------+
|member_id|first_name|last_name|facility_name  |slots|booking_amount|start_time         |
+---------+----------+---------+---------------+-----+--------------+-------------------+
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-07-09 09:00:00|
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-07-27 12:00:00|
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-07-29 12:00:00|
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-08-01 09:30:00|
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-08-07 09:00:00|
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-08-20 15:00:00|
|1        |Darren    |Smith    |Badminton Court|9    |0.0           |2022-08-28 13:30:00|
|1        |Darren    |Smith    |Badminton Court|6    |0.0           |2022-09-07 14:00:00|
|1        

In [76]:
result_df = (
    members_df.join(bookings_df, expr("m.member_id = b.member_id"), "left")
    .where("m.first_name == 'Darren' and m.last_name == 'Smith'")
    .select("m.member_id", "m.first_name", "m.last_name", "m.address","b.facility_id","b.slots")
    .orderBy(col("b.slots").desc_nulls_first())
)
result_df.show(truncate=False)


+---------+----------+---------+------------------------------+-----------+-----+
|member_id|first_name|last_name|address                       |facility_id|slots|
+---------+----------+---------+------------------------------+-----------+-----+
|37       |Darren    |Smith    |3 Funktown, Denzington, Boston|NULL       |NULL |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |9    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darren    |Smith    |8 Bloomsbury Close, Boston    |2          |6    |
|1        |Darre

In [82]:
booking_10_more_df = bookings_df.filter("slots>10")

In [84]:
result_df = (
    booking_10_more_df.join(facilities_df,booking_10_more_df.facility_id == facilities_df.facility_id, "right")
    .select(facilities_df.facility_name, facilities_df.member_cost, facilities_df.guest_cost, booking_10_more_df.start_time, booking_10_more_df.slots)
)

In [86]:
result_df.show(truncate=False)

+---------------+-----------+----------+-------------------+-----+
|facility_name  |member_cost|guest_cost|start_time         |slots|
+---------------+-----------+----------+-------------------+-----+
|Tennis Court 1 |5.0        |25.0      |2022-09-15 08:00:00|12   |
|Tennis Court 2 |5.0        |25.0      |NULL               |NULL |
|Badminton Court|0.0        |15.5      |NULL               |NULL |
|Table Tennis   |0.0        |5.0       |NULL               |NULL |
|Massage Room 1 |35.0       |80.0      |NULL               |NULL |
|Massage Room 2 |35.0       |80.0      |NULL               |NULL |
|Squash Court   |3.5        |17.5      |2022-09-13 10:30:00|14   |
|Snooker Table  |0.0        |5.0       |NULL               |NULL |
|Pool Table     |0.0        |5.0       |NULL               |NULL |
+---------------+-----------+----------+-------------------+-----+

